In [1]:
import json 
import os  

with open('config.json', 'r') as f:
    data = json.load(f)

pathway_gen = os.path.abspath(data["python_files"])
pathway_temp = os.path.abspath(data["publications"])
pathway = os.path.join(pathway_temp, "engelmann2017social")
original_data_pathway = os.path.join(pathway, "original_data")

complete_path_1 = os.path.join(original_data_pathway, "Engelmann_2017_ChIA test data.csv")

out_pathway = os.path.join(pathway, "standardized_data")
if not os.path.exists(out_pathway):
    os.makedirs(out_pathway)

In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv(complete_path_1)

df['study_id']="engelmann2017social"
df.columns = map(str.lower, df.columns)
df=df.applymap(lambda s: s.lower() if type(s) == str else s)
df = df.rename(columns={"chimp": "ape",
    "partner_name": "ape_2",
    "sex": "sex_original"})

In [3]:
df = df.replace(['/'], [np.nan])


comp_path_name_errors = os.path.join(pathway_gen, "common_name_errors.csv")

df_name  = pd.read_csv(comp_path_name_errors)

for x,y in zip(df_name['wrong'],df_name['right']):
    df['ape'].replace(x, y, inplace=True)
    df['ape_2'].replace(x, y, inplace=True)

df['dyad']=df.ape.str.cat(df.ape_2, sep='_')
# df.columns

In [4]:
role=[]
role_2=[]
for index, row in df.iterrows():
    if not pd.isna(row['ape']):
        role.append("focal_participant")
    else:
        role.append("")
df = df.assign(role=role)
for index, row in df.iterrows(): 
    if not pd.isna(row['ape_2']):
        role_2.append("partner")
    else:
        role_2.append("")
df = df.assign(role_2=role_2)


In [5]:
df[['day', 'month', 'year']] = df['date'].str.split('.', expand=True)
df['year'] = '20' + df['year'].astype(str)

In [6]:
comp_path_ape_info = os.path.join(pathway_gen, "apes_includeindatabase.csv")
apedf = pd.read_csv(comp_path_ape_info)   
df= df.merge(apedf,left_on='ape', right_on='name', how='left') 

comp_path_ape_info_2 = os.path.join(pathway_gen, "apes_includeindatabase_2.csv")
apedf_2 = pd.read_csv(comp_path_ape_info_2)
df= df.merge(apedf_2,left_on='ape_2', right_on='name_2', how='left')



In [7]:
df=df[df['ape'].notna()]

# df.columns
df.rename(columns={"ape": "participant", "ape_2":"participant_2",
                   "group":"species_subgroup",
                   'counterbalance_first_subject_main_reset':'counterbalance_first_focal_participant_main_reset'}, inplace=True)

In [8]:
comp_path_birth_dates = os.path.join(pathway_gen, "apes_age_calculations.csv")
ape_dob = pd.read_csv(comp_path_birth_dates) 
df= df.merge(ape_dob,left_on='participant', right_on='name', how='left')

comp_path_birth_dates_2 = os.path.join(pathway_gen, "apes_age_calculations_2.csv")
ape_dob_2 = pd.read_csv(comp_path_birth_dates_2) 
df= df.merge(ape_dob_2,left_on='participant_2', right_on='name_2', how='left')
two_participant_lists = [['dodc','dob','age_in_years'],
                        ['dodc_2','dob_2','age_in_years_2']]
for x,y,k in two_participant_lists:
    df[x] = df['year'].astype(str) + '-' + df['month'].astype(str) + '-' + df['day'].astype(str)
    df[x] = pd.to_datetime(df[x])
    df[y] = pd.to_datetime(df[y])
    df[k] = (df[x] - df[y]).dt.days//365

In [9]:
df.rename(columns={"result_machine": "distributor_exchange",
                    'result_refinitpass':'passive_refusal_to_initiate_trial', 
                    'result_refinitact':'active_refusal_to_initiate_trial',
                    'result_refexpass':'passive_refusal_to_exchange_for_food', 
                    'result_refexact':'active_refusal_to_exchange_for_food', 
                    'result_refconpass':'passive_refusal_to_consume_food',
                    'result_refconact':'active_refusal_to_consume_food'}, inplace=True)

In [10]:
engelmann2017social_standardized=df[['study_id',  'year', 'month', 'day',
        'participant',  'age_in_years','sex', 'role', 
        'participant_2','age_in_years_2','sex_2','role_2','species', 'dyad', 'species_subgroup', 'session', 'trial',
       'condition_machine', 'condition_partner', 'distributor_exchange',
       'result_optout', 'passive_refusal_to_initiate_trial', 'active_refusal_to_initiate_trial',
       'passive_refusal_to_exchange_for_food', 'active_refusal_to_exchange_for_food', 'passive_refusal_to_consume_food',
       'active_refusal_to_consume_food', 'counterbalance_high_value_platform',
       'counterbalance_first_focal_participant_main_reset',
       'counterbalance_partner_presence_first', 'foodtype', 'exclude'  ]]

In [11]:
comp_out_path_stand = os.path.join(out_pathway, 'engelmann2017social_standardized.csv')
engelmann2017social_standardized.to_csv(comp_out_path_stand, encoding='utf-8-sig', index=False)

In [12]:
names =engelmann2017social_standardized.columns.tolist()
df = pd.DataFrame(names)
df = df.rename(columns={0: "column_name"})
df["description"] = ""
engelmann2017social_glossary=df[["column_name", "description"]]

comp_out_path_glossary = os.path.join(out_pathway, 'engelmann2017social_glossary.csv')
engelmann2017social_glossary.to_csv(comp_out_path_glossary, encoding='utf-8-sig', index=False)